# R2AI ANDGATE Kaggle Pipeline

Notebook này dùng để chạy repo `danhyoyo/r2ai_andgate` trên Kaggle: clone repo, cài dependency, tạo corpus luật, build index, chạy test và xuất `results.json` / `submission.zip`.

Khuyến nghị trên Kaggle free: chạy `USE_LLM = False` trước để lấy kết quả chắc chắn bằng BM25 + BGE-M3 + reranker. Chỉ bật `USE_LLM = True` khi GPU đủ VRAM và còn nhiều thời gian runtime.

## 0. Kaggle Setup

Trong Kaggle Notebook, bật:

- `Accelerator`: GPU.
- `Internet`: On, nếu cần clone GitHub và tải Hugging Face model/dataset.

Nếu cuộc thi hoặc notebook tắt Internet, hãy upload repo/corpus/model thành Kaggle Dataset rồi copy từ `/kaggle/input/...`.

In [ ]:
from pathlib import Path
import glob
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/danhyoyo/r2ai_andgate.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "r2ai_andgate"

# Corpus import. Tăng IMPORT_DOCS khi muốn corpus lớn hơn; dùng -1 để quét toàn bộ nếu còn đủ thời gian.
FORCE_IMPORT = False
IMPORT_DOCS = 5000
IMPORT_CHUNK_SIZE = 1000
METADATA_SCAN_LIMIT = 50000
IMPORT_BATCH_SIZE = 100
IMPORT_BACKEND = "datasets"     # datasets backend avoids flaky datasets-server /rows 502 errors

# Inference mode.
VARIANT = "balanced"          # recall | balanced | precision
USE_DENSE = True              # BAAI/bge-m3
USE_CROSS_ENCODER = True      # BAAI/bge-reranker-v2-m3
USE_LLM = False               # Bật True nếu GPU đủ mạnh cho Qwen2.5-7B-Instruct

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_DIR = WORK_DIR / "models" / "Qwen2.5-7B-Instruct"

# Sharding giúp tránh timeout 12h. Ví dụ chạy 4 lượt: NUM_SHARDS=4, SHARD_ID lần lượt 0,1,2,3.
NUM_SHARDS = 1
SHARD_ID = 0

ARTICLES_PATH = Path("data/processed/articles.jsonl")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Config ready")

## 1. Clone repo từ GitHub

In [ ]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repo:", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

## 2. Cài dependency và kiểm tra GPU

In [ ]:
packages = [
    "numpy",
    "transformers",
    "sentence-transformers",
    "huggingface-hub",
    "datasets",
    "pyarrow",
    "accelerate",
    "safetensors",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except Exception as exc:
    print("Torch/GPU check failed:", exc)

subprocess.run([sys.executable, "-m", "src.models.check_environment"], check=True)

## 3. Chuẩn bị corpus luật

Notebook ưu tiên dùng `articles.jsonl` nếu bạn đã upload sẵn trong Kaggle Dataset. Nếu không có, nó sẽ import từ Hugging Face theo từng chunk để có thể resume tốt hơn.

In [ ]:
Path("data/processed").mkdir(parents=True, exist_ok=True)

uploaded_articles = list(Path("/kaggle/input").glob("**/articles.jsonl"))
if uploaded_articles and not FORCE_IMPORT:
    source = uploaded_articles[0]
    shutil.copy2(source, ARTICLES_PATH)
    print("Copied uploaded corpus:", source, "->", ARTICLES_PATH)
else:
    if ARTICLES_PATH.exists():
        ARTICLES_PATH.unlink()
    if IMPORT_DOCS == -1:
        cmd = [
            sys.executable, "-m", "src.preprocess.import_hf_legal",
            "--backend", IMPORT_BACKEND,
            "--output", str(ARTICLES_PATH),
            "--max-docs", "-1",
            "--metadata-scan-limit", "-1",
            "--batch-size", str(IMPORT_BATCH_SIZE),
            "--require-metadata",
        ]
        print("Running full import:", " ".join(cmd))
        subprocess.run(cmd, check=True)
    else:
        for offset in range(0, IMPORT_DOCS, IMPORT_CHUNK_SIZE):
            max_docs = min(IMPORT_CHUNK_SIZE, IMPORT_DOCS - offset)
            cmd = [
                sys.executable, "-m", "src.preprocess.import_hf_legal",
                "--backend", IMPORT_BACKEND,
                "--output", str(ARTICLES_PATH),
                "--offset", str(offset),
                "--max-docs", str(max_docs),
                "--metadata-scan-limit", str(METADATA_SCAN_LIMIT),
                "--batch-size", str(IMPORT_BATCH_SIZE),
                "--require-metadata",
                "--append",
            ]
            print("Import chunk offset", offset, "size", max_docs)
            subprocess.run(cmd, check=True)

assert ARTICLES_PATH.exists(), "Missing data/processed/articles.jsonl"
line_count = sum(1 for _ in ARTICLES_PATH.open("r", encoding="utf-8"))
print("Article rows:", line_count)
print("Corpus path:", ARTICLES_PATH.resolve())

## 4. Build BM25 index

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.retrieval.bm25_retriever", "build",
    "--articles", str(ARTICLES_PATH),
    "--output", "indexes/bm25/bm25.pkl",
], check=True)

## 5. Smoke test retrieval

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.retrieval.pipeline",
    "--articles", str(ARTICLES_PATH),
    "--question", "Doanh nghiệp nhỏ và vừa được hỗ trợ khi đáp ứng điều kiện nào?",
    "--variant", VARIANT,
], check=True)

## 6. Tạo test shard để tránh timeout

Nếu sợ hết 12 tiếng, đặt `NUM_SHARDS = 4` rồi chạy 4 session với `SHARD_ID = 0, 1, 2, 3`. Mỗi session tạo một file `results_shard_*`. Sau đó dùng cell merge ở cuối.

In [ ]:
test_records = json.loads(Path("data/test/test.json").read_text(encoding="utf-8"))
assert 0 <= SHARD_ID < NUM_SHARDS

if NUM_SHARDS > 1:
    shard_records = [row for index, row in enumerate(test_records) if index % NUM_SHARDS == SHARD_ID]
    test_path = Path(f"data/test/test_shard_{SHARD_ID}_of_{NUM_SHARDS}.json")
    test_path.write_text(json.dumps(shard_records, ensure_ascii=False, indent=2), encoding="utf-8")
    output_path = RESULTS_DIR / f"results_shard_{SHARD_ID}_of_{NUM_SHARDS}.json"
else:
    test_path = Path("data/test/test.json")
    output_path = RESULTS_DIR / "results.json"

print("Test path:", test_path, "records:", len(json.loads(test_path.read_text(encoding="utf-8"))))
print("Output path:", output_path)

## 7. Optional: tải Qwen local

Chỉ chạy cell này khi `USE_LLM = True`. Trên GPU 16GB, khả năng OOM cao; trên 24GB sẽ ổn hơn.

In [ ]:
model_args = []
if USE_LLM:
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "huggingface-cli", "download", MODEL_ID,
        "--local-dir", str(MODEL_DIR),
    ], check=True)
    model_args = ["--model-path", str(MODEL_DIR)]
    print("Using local LLM:", MODEL_DIR)
else:
    print("USE_LLM=False: using extractive grounded answer fallback.")

## 8. Chạy inference tạo kết quả

In [ ]:
cmd = [
    sys.executable, "-m", "src.submit.build_results",
    "--test", str(test_path),
    "--articles", str(ARTICLES_PATH),
    "--output", str(output_path),
    "--variant", VARIANT,
]
if USE_DENSE:
    cmd.append("--use-dense")
if USE_CROSS_ENCODER:
    cmd.append("--use-cross-encoder")
cmd.extend(model_args)

print("Running:", " ".join(cmd))
start = time.time()
subprocess.run(cmd, check=True)
print("Done in minutes:", round((time.time() - start) / 60, 2))

## 9. Validate và zip

Nếu chạy shard thì mỗi shard chỉ validate file shard. Chỉ zip submission cuối khi đã merge đủ shard.

In [ ]:
subprocess.run([sys.executable, "-m", "src.submit.validate_results", "--input", str(output_path), "--strict"], check=True)

if NUM_SHARDS == 1:
    subprocess.run([
        sys.executable, "-m", "src.submit.zip_submission",
        "--results", str(output_path),
        "--output", "results/submission.zip",
    ], check=True)
    print("Final files:", output_path, Path("results/submission.zip"))
else:
    print("Shard file is ready:", output_path)
    print("Run all shard IDs, then merge them with the next cell.")

## 10. Merge shard outputs nếu chia nhiều session

Copy tất cả `results_shard_*_of_*.json` vào thư mục `results/` rồi chạy cell này.

In [ ]:
shard_paths = sorted(Path("results").glob("results_shard_*_of_*.json"))
if shard_paths:
    merged = []
    seen_ids = set()
    for path in shard_paths:
        rows = json.loads(path.read_text(encoding="utf-8"))
        for row in rows:
            if row["id"] not in seen_ids:
                seen_ids.add(row["id"])
                merged.append(row)
    merged.sort(key=lambda item: item["id"])
    final_path = Path("results/results.json")
    final_path.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Merged records:", len(merged), "->", final_path)
    subprocess.run([sys.executable, "-m", "src.submit.validate_results", "--input", str(final_path), "--strict"], check=True)
    subprocess.run([sys.executable, "-m", "src.submit.zip_submission", "--results", str(final_path), "--output", "results/submission.zip"], check=True)
else:
    print("No shard files found.")

## 11. Xem nhanh output

In [ ]:
for path in [Path("results/results.json"), Path("results/submission.zip"), output_path, ARTICLES_PATH]:
    if path.exists():
        print(path, round(path.stat().st_size / 1024 / 1024, 2), "MB")

preview_path = Path("results/results.json") if Path("results/results.json").exists() else output_path
preview = json.loads(preview_path.read_text(encoding="utf-8"))[:2]
preview